In [22]:
# --- CELL A: GUI LAYOUT WITH DETAILED COMPENSATION TOGGLES ---
import ipywidgets as widgets
import pandas as pd

lbl_style = widgets.Layout(width='125px')
DEDUCTION_DEFAULTS = {'Single': 16100.00, 'Married Filing Jointly': 32200.00}

status_dropdown = widgets.Dropdown(options=['Single', 'Married Filing Jointly'], value='Single', description='Filing Status:')
apply_deduction_check = widgets.Checkbox(value=False, description='Apply Standard Deduction Reduction')
date_picker = widgets.DatePicker(value=pd.to_datetime('2026-06-15').date(), description='Pay Date:')

# Pair 1: Annual Base Salary
salary_lbl = widgets.Label('Annual Base:', layout=lbl_style)
salary_slide = widgets.IntSlider(value=60000, min=30000, max=250000, step=1000, continuous_update=False)
salary_box = widgets.IntText(value=60000, layout=widgets.Layout(width='90px'))
widgets.jslink((salary_slide, 'value'), (salary_box, 'value'))
salary_ui = widgets.HBox([salary_lbl, salary_slide, salary_box])

# Pair 2: Pay Periods
periods_lbl = widgets.Label('Pay Periods:', layout=lbl_style)
periods_slide = widgets.IntSlider(value=12, min=12, max=52, step=1, continuous_update=False)
periods_box = widgets.IntText(value=12, layout=widgets.Layout(width='90px'))
widgets.jslink((periods_slide, 'value'), (periods_box, 'value'))
periods_ui = widgets.HBox([periods_lbl, periods_slide, periods_box])

# Pair 3: S-Corp HIP + Payment Type Dropdown
hip_lbl = widgets.Label('S-Corp HIP:', layout=lbl_style)
hip_slide = widgets.FloatSlider(value=1310.0, min=0.0, max=2000.0, step=10.0, continuous_update=False)
hip_box = widgets.FloatText(value=1310.0, layout=widgets.Layout(width='90px'))
widgets.jslink((hip_slide, 'value'), (hip_box, 'value'))
hip_toggle = widgets.Dropdown(options=['Direct Paid by ER', 'Reimbursement to EE'], value='Direct Paid by ER', layout=widgets.Layout(width='150px'))
hip_ui = widgets.HBox([hip_lbl, hip_slide, hip_box, hip_toggle])

# Pair 4: S-Corp DIP + Payment Type Dropdown
dip_lbl = widgets.Label('S-Corp DIP:', layout=lbl_style)
dip_slide = widgets.FloatSlider(value=0.0, min=0.0, max=1000.0, step=5.0, continuous_update=False)
dip_box = widgets.FloatText(value=0.0, layout=widgets.Layout(width='90px'))
widgets.jslink((dip_slide, 'value'), (dip_box, 'value'))
dip_toggle = widgets.Dropdown(options=['Direct Paid by ER', 'Reimbursement to EE'], value='Direct Paid by ER', layout=widgets.Layout(width='150px'))
dip_ui = widgets.HBox([dip_lbl, dip_slide, dip_box, dip_toggle])

# Pair 5: 401k Deferral
retire_lbl = widgets.Label('401k Def %:', layout=lbl_style)
retire_slide = widgets.FloatSlider(value=0.0, min=0.0, max=25.0, step=0.5, continuous_update=False)
retire_box = widgets.FloatText(value=0.0, layout=widgets.Layout(width='90px'))
widgets.jslink((retire_slide, 'value'), (retire_box, 'value'))
retire_ui = widgets.HBox([retire_lbl, retire_slide, retire_box])

# Pair 6: Standard Deduction Override
deduct_lbl = widgets.Label('Std Deduction:', layout=lbl_style)
deduct_slide = widgets.FloatSlider(value=16100.0, min=0.0, max=45000.0, step=100.0, continuous_update=False)
deduct_box = widgets.FloatText(value=16100.0, layout=widgets.Layout(width='90px'))

def sync_deduct_slider_to_box(change): deduct_box.value = change['new']
def sync_deduct_box_to_slider(change):
    val = change['new']
    if val is None or str(val).strip() == "": deduct_slide.value = 0.0
    else:
        if val > deduct_slide.max: deduct_slide.max = val * 1.2
        deduct_slide.value = val

deduct_slide.observe(sync_deduct_slider_to_box, names='value')
deduct_box.observe(sync_deduct_box_to_slider, names='value')
deduct_ui = widgets.HBox([deduct_lbl, deduct_slide, deduct_box])

def on_status_changed(change):
    if deduct_slide.value in [None, 0.0, 16100.0, 32200.0]:
        deduct_slide.value = DEDUCTION_DEFAULTS[change['new']]

status_dropdown.observe(on_status_changed, 'value')
save_button = widgets.Button(description="Commit Pay Run", button_style='success', icon='check')
output_panel = widgets.Output()

input_form = widgets.VBox([status_dropdown, apply_deduction_check, deduct_ui, salary_ui, periods_ui, hip_ui, dip_ui, retire_ui, date_picker, save_button])
display(widgets.HBox([input_form, output_panel]))


In [23]:
# --- CELL B: MASTER MATH ENGINE & VISUAL DASHBOARD ---
import sqlite3
import matplotlib.pyplot as plt
from IPython.display import display, clear_output


DB_NAME, CSV_BACKUP_NAME, EMPLOYEE_ID = "payroll_ledger.db", "payroll_history_backup.csv", "EE-SCORP-01"
SS_WAGE_BASE, FUTA_WAGE_BASE, NM_SUI_WAGE_BASE = 184500.00, 7000.00, 34800.00   
SS_RATE, MEDICARE_RATE, FUTA_NET_RATE, NM_SUI_RATE = 0.062, 0.0145, 0.006, 0.034

TAX_TABLES_2026 = {
    'Single': {
        'FED': [(12400, 0.10), (50400, 0.12), (105700, 0.22), (201775, 0.24), (256225, 0.32), (640600, 0.35), (float('inf'), 0.37)],
        'NM':  [(8050, 0.00), (13550, 0.015), (20550, 0.032), (24550, 0.032), (33000, 0.043), (41000, 0.043), (58000, 0.047), (74000, 0.047), (102100, 0.047), (116100, 0.049), (217500, 0.049), (331100, 0.049), (float('inf'), 0.059)]
    },
    'Married Filing Jointly': {
        'FED': [(24800, 0.10), (100800, 0.12), (211400, 0.22), (403550, 0.24), (512450, 0.32), (768700, 0.35), (float('inf'), 0.37)],
        'NM':  [(16100, 0.00), (27100, 0.015), (41100, 0.032), (49100, 0.032), (66000, 0.043), (82000, 0.043), (116000, 0.047), (148000, 0.047), (204200, 0.047), (232200, 0.049), (435000, 0.049), (662200, 0.049), (float('inf'), 0.059)]
    }
}

def calc_capped(gross, ytd, cap, rate):
    return 0.0 if ytd >= cap else min(gross, cap - ytd) * rate

def calc_progressive_adjusted(gross, periods, brackets, apply_deduction, deduction_val):
    annual_taxable = max(0.00, (gross * periods) - deduction_val) if apply_deduction else (gross * periods)
    tax_owed = 0.0; prev = 0.0
    for limit, rate in brackets:
        if annual_taxable > limit:
            tax_owed += (limit - prev) * rate; prev = limit
        else:
            tax_owed += (annual_taxable - prev) * rate; break
    return tax_owed / periods

# --- 🚀 THE UNIFIED CENTRAL CALCULATOR FUNCTION 🚀 ---
def calculate_payroll_metrics(status, periods, annual_salary, hip, dip, retire_pct, apply_deduct, deduct_val, ytd_prior):
    base_gross = annual_salary / periods
    total_payout = base_gross + hip + dip
    deduction_401k = base_gross * (retire_pct / 100.0)
    
    ee_ss = calc_capped(base_gross, ytd_prior, SS_WAGE_BASE, SS_RATE)
    ee_med = base_gross * MEDICARE_RATE
    combined_tax_gross = base_gross + hip + dip - deduction_401k
    
    ee_fed = calc_progressive_adjusted(combined_tax_gross, periods, TAX_TABLES_2026[status]['FED'], apply_deduct, deduct_val)
    ee_nm = calc_progressive_adjusted(combined_tax_gross, periods, TAX_TABLES_2026[status]['NM'], apply_deduct, deduct_val)
    nm_wc_fee = (2.55 * 4) / periods
    
    total_ee_taxes = ee_ss + ee_med + ee_fed + ee_nm + nm_wc_fee
    net_pay = total_payout - deduction_401k - total_ee_taxes
    
    er_ss, er_med = ee_ss, ee_med
    er_futa = calc_capped(base_gross, ytd_prior, FUTA_WAGE_BASE, FUTA_NET_RATE)
    er_suta = calc_capped(base_gross, ytd_prior, NM_SUI_WAGE_BASE, NM_SUI_RATE)
    total_er_taxes = er_ss + er_med + er_futa + er_suta + nm_wc_fee
    total_company_cost = total_payout + total_er_taxes

    return {
        'base_gross': base_gross, 'total_payout': total_payout, 'deduction_401k': deduction_401k,
        'ee_ss': ee_ss, 'ee_med': ee_med, 'ee_fed': ee_fed, 'ee_nm': ee_nm, 'nm_wc_fee': nm_wc_fee,
        'net_pay': net_pay, 'er_futa': er_futa, 'er_suta': er_suta, 'total_er_taxes': total_er_taxes, 'total_company_cost': total_company_cost
    }

def update_dashboard(*args):
    with output_panel:
        clear_output(wait=True)
        conn = sqlite3.connect(DB_NAME); cursor = conn.cursor()
        # 🔄 MINIMAL REPAIR: Safely initialize the missing schema if the database was purged
        cursor.execute("""
        CREATE TABLE IF NOT EXISTS payroll_ledger (
            id INTEGER PRIMARY KEY AUTOINCREMENT, employee_id TEXT, pay_period_ending TEXT, gross_wages REAL, pre_tax_401k REAL,
            ee_ss REAL, ee_medicare REAL, ee_fed_tax REAL, ee_nm_tax REAL, ee_wc_fee REAL, net_pay REAL,
            er_ss REAL, er_medicare REAL, er_futa REAL, er_nm_sui REAL, er_wc_fee REAL, total_company_cost REAL,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
        )""")
        cursor.execute("SELECT SUM(gross_wages) FROM payroll_ledger WHERE employee_id = ?", (EMPLOYEE_ID,))
        res = cursor.fetchone(); ytd_prior = float(res[0]) if res and res[0] is not None else 0.00; conn.close()
        
        # Call the unified function
        m = calculate_payroll_metrics(
            status_dropdown.value, periods_slide.value, salary_slide.value, 
            hip_slide.value, dip_slide.value, retire_slide.value, 
            apply_deduction_check.value, deduct_slide.value, ytd_prior
        )
        
        # --- SEGMENTED CASH FLOW DISTRIBUTION ---
        pure_labor_check = m['base_gross'] - m['deduction_401k'] - (m['ee_ss'] + m['ee_med'] + m['ee_fed'] + m['ee_nm'] + m['nm_wc_fee'])
        reimbursement_cash = sum([hip_slide.value if hip_toggle.value == 'Reimbursement to EE' else 0, dip_slide.value if dip_toggle.value == 'Reimbursement to EE' else 0])
        direct_paid_insurance = sum([hip_slide.value if hip_toggle.value == 'Direct Paid by ER' else 0, dip_slide.value if dip_toggle.value == 'Direct Paid by ER' else 0])
        
        # Visual Canvas Drawing
        sizes = [max(0, pure_labor_check), m['deduction_401k'], m['ee_fed'], m['ee_nm'], (m['ee_ss'] + m['ee_med']), direct_paid_insurance, reimbursement_cash]
        names = ['Labor Check Base', '401k Def', 'Fed Tax', 'NM Tax', 'FICA taxes', 'ER Direct Insurance', 'EE Cash Reimburse']
        colors = ['#2ecc71', '#3498db', '#e74c3c', '#f1c40f', '#95a5a6', '#9b59b6', '#e67e22']
        combined_labels = [f"{n}\n(${v:,.2f})" for n, v in zip(names, sizes)]
        labels = [l for i, l in enumerate(combined_labels) if sizes[i] > 0]
        colors = [c for i, c in enumerate(colors) if sizes[i] > 0]
        sizes = [s for s in sizes if s > 0]
        
        fig, ax = plt.subplots(figsize=(8.5, 4.0))
        ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=140, pctdistance=0.65, labeldistance=1.15, textprops={'fontsize': 9, 'weight': 'bold'})
        ax.set_title("S-Corp Complete Package Allocation", fontsize=11, weight='bold', pad=20)
        plt.tight_layout(); plt.show()
        
        print("=" * 65)
        print("        S-CORP BOOKKEEPING DISBURSEMENT LOG SUMMARY       ")
        print("=" * 65)
        print(f"📄 Total Shareholder Compensation Value Package:  ${(m['total_payout'] - m['deduction_401k'] - (m['ee_ss'] + m['ee_med'] + m['ee_fed'] + m['ee_nm'] + m['nm_wc_fee'])):10,.2f}")
        print("-" * 65)
        print("💵 INDEPENDENT PHYSICAL DISBURSEMENT TRANSACTIONS:")
        print(f"   👉 CHECK #1 (WAGES VOUCHER):               ${pure_labor_check:10,.2f}")
        print(f"   👉 CHECK #2 (REIMBURSEMENT VOUCHER):        ${reimbursement_cash:10,.2f}")
        print(f"   👉 BANK WIRE #3 (DIRECT CARRIER TRANSFER):  ${direct_paid_insurance:10,.2f}")
        print("-" * 65)
        print("💼 EMPLOYER-SIDE STATUTORY OVERHEAD TAX LIABILITIES")
        print(f"   [+] Matching Social Security (6.2%):  ${m['ee_ss']:10,.2f}")
        print(f"   [+] Matching Medicare (1.45%):        ${m['ee_med']:10,.2f}")
        print(f"   [+] Federal Unemployment (FUTA 0.6%): ${m['er_futa']:10,.2f}")
        print(f"   [+] NM State Unemployment (SUTA 3.4%):${m['er_suta']:10,.2f}")
        print("-" * 65)
        print(f"📊 TOTAL EMPLOYER-SIDE PAYROLL TAXES:   ${m['total_er_taxes']:10,.2f}")
        print(f"🏢 TOTAL TRANSACTION COMPANY COST:       ${m['total_company_cost']:10,.2f}")
        print("=" * 65)

for w in [status_dropdown, apply_deduction_check, salary_slide, periods_slide, hip_slide, dip_slide, retire_slide, deduct_slide, date_picker, hip_toggle, dip_toggle]:
    w.observe(update_dashboard, 'value')
update_dashboard()


In [24]:
# --- CELL C: CLEAN DATABASE PERSISTENCE ENGINE ---
def execute_db_append(payload):
    conn = sqlite3.connect(DB_NAME); cursor = conn.cursor()
    cols = "employee_id, pay_period_ending, gross_wages, pre_tax_401k, ee_ss, ee_medicare, ee_fed_tax, ee_nm_tax, ee_wc_fee, net_pay, er_ss, er_medicare, er_futa, er_nm_sui, er_wc_fee, total_company_cost"
    placeholders = ", ".join(["?"] * 16)
    cursor.execute(f"INSERT INTO payroll_ledger ({cols}) VALUES ({placeholders})", payload)
    conn.commit()
    df = pd.read_sql_query("SELECT * FROM payroll_ledger WHERE employee_id = ?", conn, params=[EMPLOYEE_ID])
    df.to_csv(CSV_BACKUP_NAME, index=False); conn.close()

def on_save_clicked(b):
    conn = sqlite3.connect(DB_NAME); cursor = conn.cursor()
    cursor.execute("SELECT SUM(gross_wages) FROM payroll_ledger WHERE employee_id = ?", (EMPLOYEE_ID,))
    res = cursor.fetchone(); ytd_prior = float(res) if res and res is not None else 0.00; conn.close()
    
    # Call the central calculation engine housed in Cell B
    m = calculate_payroll_metrics(
        status_dropdown.value, periods_slide.value, salary_slide.value, 
        hip_slide.value, dip_slide.value, retire_slide.value, 
        apply_deduction_check.value, deduct_slide.value, ytd_prior
    )
    
    # Package values cleanly for the database insert statement
    current_gross = salary_slide.value / periods_slide.value + hip_slide.value + dip_slide.value
    data_payload = (
        EMPLOYEE_ID, str(date_picker.value), current_gross, m['deduction_401k'], 
        m['ee_ss'], m['ee_med'], m['ee_fed'], m['ee_nm'], m['nm_wc_fee'], m['net_pay'], 
        m['ee_ss'], m['ee_med'], m['er_futa'], m['er_suta'], m['nm_wc_fee'], m['total_company_cost']
    )
    
    execute_db_append(data_payload)
    update_dashboard()
    with output_panel: print(f"\n✅ SUCCESS: Payroll recorded cleanly using Central Math Engine data.")

save_button.on_click(on_save_clicked)
print("Database engine linked directly to centralized master math function.")


Database engine linked directly to centralized master math function.


In [25]:
# --- CELL D: LIVE YEAR-END W-2 COMPLIANCE FORECAST ENGINE ---
def run_w2_forecast():
    conn = sqlite3.connect(DB_NAME)
    df = pd.read_sql_query("SELECT * FROM payroll_ledger WHERE employee_id = ?", conn, params=[EMPLOYEE_ID])
    conn.close()
    
    # 1. Gather historical baseline statistics from database
    historical_runs = len(df)
    ytd_401k = df['pre_tax_401k'].sum() if historical_runs > 0 else 0.00
    ytd_fed_withheld = df['ee_fed_tax'].sum() if historical_runs > 0 else 0.00
    ytd_nm_withheld = df['ee_nm_tax'].sum() if historical_runs > 0 else 0.00
    ytd_ss_withheld = df['ee_ss'].sum() if historical_runs > 0 else 0.00
    ytd_med_withheld = df['ee_medicare'].sum() if historical_runs > 0 else 0.00
    
    # Track the historical wage bases from the database
    # For Box 3 and 5, we look strictly at the FICA-subject base salary component
    if historical_runs > 0:
        # Total Gross recorded includes HIP/DIP, so we back out the recorded insurance to isolate true base
        # To make it even simpler, we can sum the base salary allocations directly
        total_periods_logged = historical_runs
        ytd_base_salary_only = (salary_slide.value / periods_slide.value) * total_periods_logged
        ytd_total_gross_with_insurance = df['gross_wages'].sum()
    else:
        ytd_base_salary_only = 0.00
        ytd_total_gross_with_insurance = 0.00

    # 2. Extract current dashboard sliders to model subsequent remaining pay runs
    total_configured_periods = periods_slide.value
    remaining_periods = max(0, total_configured_periods - historical_runs)
    
    base_salary_per_period = salary_slide.value / total_configured_periods
    hip_period, dip_period = hip_slide.value, dip_slide.value
    deduction_401k_period = base_salary_per_period * (retire_slide.value / 100.0)
    
    # Projection variables
    future_projected_base_salary = base_salary_per_period * remaining_periods
    future_projected_insurance = (hip_period + dip_period) * remaining_periods
    future_projected_401k = deduction_401k_period * remaining_periods
    
    # Run a clean math simulation for progressive text withholdings over remaining periods
    projected_fed = 0.0
    projected_nm = 0.0
    simulated_ytd_gross_for_caps = ytd_base_salary_only # SS/FUTA/SUTA caps trace base salary
    
    for _ in range(remaining_periods):
        # Income tax is subject to base + insurance - 401k
        tax_subject_period = base_salary_per_period + hip_period + dip_period - deduction_401k_period
        
        ee_fed = calc_progressive_adjusted(tax_subject_period, total_configured_periods, TAX_TABLES_2026[status_dropdown.value]['FED'], apply_deduction_check.value, deduct_slide.value)
        ee_nm = calc_progressive_adjusted(tax_subject_period, total_configured_periods, TAX_TABLES_2026[status_dropdown.value]['NM'], apply_deduction_check.value, deduct_slide.value)
        
        projected_fed += ee_fed
        projected_nm += ee_nm
        simulated_ytd_gross_for_caps += base_salary_per_period

    # 3. Compile Total Projected Year-End W-2 Fields (YTD + Future Projected Runs)
    annualized_base_salary = ytd_base_salary_only + future_projected_base_salary
    annualized_insurance_premiums = ((hip_period + dip_period) * total_configured_periods)
    annualized_401k_deferred = ytd_401k + future_projected_401k
    
    # --- ⚖️ RECONFIGURED S-CORP W-2 BOX SPECIFICATIONS ⚖️ ---
    # Box 1: Includes regular base salary + company-paid health insurance MINUS pre-tax retirement
    box_1_wages = annualized_base_salary + annualized_insurance_premiums - annualized_401k_deferred
    
    # Box 3: Base salary only, capped strictly at the 2026 limit ($184,500)
    box_3_wages = min(annualized_base_salary, SS_WAGE_BASE)
    
    # Box 5: Base salary only, completely uncapped (no matter how high the slider moves)
    box_5_wages = annualized_base_salary
    
    # Projected final tax totals
    final_fed_withheld = ytd_fed_withheld + projected_fed
    final_nm_withheld = ytd_nm_withheld + projected_nm
    
    # Calculate exact FICA tracking based on base salary rules
    final_ss_withheld = min(annualized_base_salary, SS_WAGE_BASE) * SS_RATE
    final_med_withheld = annualized_base_salary * MEDICARE_RATE
    
    print("=" * 65)
    print(f"      CORRECTED 2026 W-2 DISCLOSURE SUMMARY ({EMPLOYEE_ID})      ")
    print("=" * 65)
    print(f" 📦 Tracked Runs inside Ledger: {historical_runs} | 🔮 Remaining Simulated Runs: {remaining_periods}")
    print("-" * 65)
    print(f" 🔹 Box 1 (Wages, Tips, Other Comp):         ${box_1_wages:11,.2f}")
    print(f"     [Formula: Base Salary (${annualized_base_salary:,.2f}) + Insurance (${annualized_insurance_premiums:,.2f}) - 401k (${annualized_401k_deferred:,.2f})]")
    print(f" 🔹 Box 2 (Federal Income Tax Withheld):      ${final_fed_withheld:11,.2f}")
    print(f" 🔹 Box 3 (Social Security Wages):           ${box_3_wages:11,.2f} (Cap: $184,500)")
    print(f"     [Strictly Excludes S-Corp Health/Dental Benefits]")
    print(f" 🔹 Box 4 (Social Security Tax Withheld):    ${final_ss_withheld:11,.2f}")
    print(f" 🔹 Box 5 (Medicare Wages and Tips):          ${box_5_wages:11,.2f} (No Cap)")
    print(f" 🔹 Box 6 (Medicare Tax Withheld):           ${final_med_withheld:11,.2f}")
    print(f" 🔹 Box 12a (Code D - 401k Deferrals):       ${annualized_401k_deferred:11,.2f}")
    print(f" 🔹 Box 14 Box 14 (S-CORP OWNER H&W INS):    ${annualized_insurance_premiums:11,.2f}")
    print(f" 🔹 Box 16 (New Mexico State Wages):         ${box_1_wages:11,.2f}")
    print(f" 🔹 Box 17 (New Mexico State Income Tax):     ${final_nm_withheld:11,.2f}")
    print("=" * 65)

# Trigger initial forecasting display update
run_w2_forecast()


      CORRECTED 2026 W-2 DISCLOSURE SUMMARY (EE-SCORP-01)      
 📦 Tracked Runs inside Ledger: 0 | 🔮 Remaining Simulated Runs: 12
-----------------------------------------------------------------
 🔹 Box 1 (Wages, Tips, Other Comp):         $  75,720.00
     [Formula: Base Salary ($60,000.00) + Insurance ($15,720.00) - 401k ($0.00)]
 🔹 Box 2 (Federal Income Tax Withheld):      $  11,370.40
 🔹 Box 3 (Social Security Wages):           $  60,000.00 (Cap: $184,500)
     [Strictly Excludes S-Corp Health/Dental Benefits]
 🔹 Box 4 (Social Security Tax Withheld):    $   3,720.00
 🔹 Box 5 (Medicare Wages and Tips):          $  60,000.00 (No Cap)
 🔹 Box 6 (Medicare Tax Withheld):           $     870.00
 🔹 Box 12a (Code D - 401k Deferrals):       $       0.00
 🔹 Box 14 Box 14 (S-CORP OWNER H&W INS):    $  15,720.00
 🔹 Box 16 (New Mexico State Wages):         $  75,720.00
 🔹 Box 17 (New Mexico State Income Tax):     $   2,773.69


In [26]:
# --- CELL E: SECURE DATABASE PURGE & LEVERAGE RESET UTILITY ---
import sqlite3
import os

def purge_payroll_database(execute_wipe=False):
    """Safely drops the ledger table and clears associated CSV flat files."""
    db_file = "payroll_ledger.db"
    csv_file = "payroll_history_backup.csv"
    
    if not execute_wipe:
        print("🛡️ SAFE PREVIEW MODE ACTIVE: No data was altered.")
        print(" -> To completely erase all historical payroll records, execute:")
        print("    purge_payroll_database(execute_wipe=True)")
        return

    # 1. Clear relational SQLite layout
    if os.path.exists(db_file):
        conn = sqlite3.connect(db_file)
        cursor = conn.cursor()
        cursor.execute("DROP TABLE IF EXISTS payroll_ledger")
        conn.commit()
        conn.close()
        print(f"💥 SUCCESS: Relational table dropped from database: '{db_file}'")
    
    # 2. Clear flat CSV backup
    if os.path.exists(csv_file):
        os.remove(csv_file)
        print(f"💥 SUCCESS: Flat backup log removed: '{csv_file}'")
        
    print("\n🔄 SYSTEM RESET COMPLETE: Run Cell A/B to initialize a fresh, empty ledger.")

# Change to True ONLY when you are ready to completely erase all historical logs
purge_payroll_database(execute_wipe=False)


🛡️ SAFE PREVIEW MODE ACTIVE: No data was altered.
 -> To completely erase all historical payroll records, execute:
    purge_payroll_database(execute_wipe=True)


In [27]:
# --- CELL F: FORM 941 QUARTERLY TAX COMPLIANCE SUMMARY ---
import sqlite3
import pandas as pd

def generate_form_941_quarterly_report():
    conn = sqlite3.connect("payroll_ledger.db")
    df = pd.read_sql_query("SELECT * FROM payroll_ledger WHERE employee_id = 'EE-SCORP-01'", conn)
    conn.close()
    
    if len(df) == 0:
        print("ℹ️ No payroll runs found in the database. Run and commit data to view reporting.")
        return
        
    # Convert string timestamps to datetime indices to group transactions by quarter
    df['pay_period_ending'] = pd.to_datetime(df['pay_period_ending'])
    df['Quarter'] = df['pay_period_ending'].dt.to_period('Q')
    
    print("=" * 65)
    print("       HISTORICAL FORM 941 QUARTERLY ACCOUNTING LEDGER      ")
    print("=" * 65)
    
    for quarter, group in df.groupby('Quarter'):
        # For Form 941 compliance purposes, we must isolate pure base wages subject to tax parameters
        # Total recorded gross minus logged health/dental benefits = true FICA/FIT base wages
        q_gross_total = group['gross_wages'].sum()
        
        # Pull parameters dynamically from matching fields
        q_401k = group['pre_tax_401k'].sum()
        q_fed_withheld = group['ee_fed_tax'].sum()
        q_ee_ss = group['ee_ss'].sum()
        q_ee_med = group['ee_medicare'].sum()
        
        # Calculate matching employer portions
        q_er_ss = q_ee_ss
        q_er_med = q_ee_med
        
        # Form 941 Specific Line Mapping Rules
        line_2_wages = q_gross_total - (1310.00 + dip_slide.value) * len(group) # Base wages subject to FIT
        line_3_fit = q_fed_withheld
        line_5a_col1 = q_gross_total - (1310.00 + dip_slide.value) * len(group) # Base wages subject to SS
        line_5a_col2 = q_ee_ss + q_er_ss
        line_5c_col1 = line_5a_col1                                            # Base wages subject to Med
        line_5c_col2 = q_ee_med + q_er_med
        
        total_taxes_before_adjustments = line_3_fit + line_5a_col2 + line_5c_col2
        
        print(f"📅 TAX REPORTING QUARTER: {quarter}")
        print(f" 🔹 Line 2: Wages, Tips, and Other Compensation: ${line_2_wages:11,.2f}")
        print(f" 🔹 Line 3: Federal Income Tax Withheld:        ${line_3_fit:11,.2f}")
        print(f" 🔹 Line 5a: Taxable Social Security Wages:     ${line_5a_col1:11,.2f} | Tax: ${line_5a_col2:,.2f}")
        print(f" 🔹 Line 5c: Taxable Medicare Wages and Tips:   ${line_5c_col1:11,.2f} | Tax: ${line_5c_col2:,.2f}")
        print(f" 🔹 Line 6: Total Taxes Before Adjustments:     ${total_taxes_before_adjustments:11,.2f}")
        print("-" * 65)

generate_form_941_quarterly_report()


ℹ️ No payroll runs found in the database. Run and commit data to view reporting.


In [28]:
# --- CELL G: FORM 940 ANNUAL FUTA LIABILITY MODULE ---
def generate_form_940_annual_report():
    conn = sqlite3.connect("payroll_ledger.db")
    df = pd.read_sql_query("SELECT * FROM payroll_ledger WHERE employee_id = 'EE-SCORP-01'", conn)
    conn.close()
    
    if len(df) == 0:
        print("ℹ️ No annual data detected. Commit payroll milestones to forecast FUTA liabilities.")
        return
        
    df['pay_period_ending'] = pd.to_datetime(df['pay_period_ending'])
    df['Year'] = df['pay_period_ending'].dt.year
    
    print("=" * 65)
    print("       ANNUAL FORM 940 FUTA UNEMPLOYMENT COMPLIANCE SUMMARY     ")
    print("=" * 65)
    
    for year, group in df.groupby('Year'):
        # Extract aggregate metrics
        total_gross_recorded = group['gross_wages'].sum()
        total_futa_tax_paid = group['er_futa'].sum()
        
        # Calculate explicit S-Corp benefit exclusions (Exempt from FUTA rules under Code Sec 3121)
        total_insurance_exemptions = (hip_slide.value + dip_slide.value) * len(group)
        total_subject_base_wages = total_gross_recorded - total_insurance_exemptions
        
        # Determine allocations exceeding the standard $7,000 threshold limit cap
        wages_in_excess_of_cap = max(0.00, total_subject_base_wages - 7000.00)
        subtotal_taxable_futa_wages = min(total_subject_base_wages, 7000.00)
        
        print(f"📅 TAX COMPLIANCE YEAR: {year}")
        print(f" 🔹 Part 2, Line 3: Total Payments to All Employees:   ${total_gross_recorded:11,.2f}")
        print(f" 🔹 Part 2, Line 4: Payments Exempt from FUTA Tax:    ${total_insurance_exemptions:11,.2f}")
        print(f"     [Includes Explicit Code Sec 3121 Owner H&W Insurance Benefits]")
        print(f" 🔹 Part 2, Line 5: Total Wages in Excess of $7,000 Cap: ${wages_in_excess_of_cap:11,.2f}")
        print(f" 🔹 Part 2, Line 7: Total Taxable FUTA Wages:           ${subtotal_taxable_futa_wages:11,.2f}")
        print(f" 🔹 Part 2, Line 8: Total FUTA Tax Before Adjustments:  ${total_futa_tax_paid:11,.2f} (@ 0.6%)")
        print("-" * 65)

generate_form_940_annual_report()


ℹ️ No annual data detected. Commit payroll milestones to forecast FUTA liabilities.
